# CFP-FP-QA01 — Original-bin quality and baseline impact audit

**Purpose:** check the 55 near-uniform review crops directly in the exact original R4 bin, then quantify how pairs containing flagged crops contribute to the frozen baseline. This is a dataset-quality experiment, not E05 attacker evaluation.

## Ready-to-run Kaggle instructions
Attach only the **CFP-FP-E01-R4 aligned embedding-cache ZIP** (or extracted directory). Enable Internet to restore the pinned baseline bin; CPU is sufficient. If you already attached that bin, set BIN_INPUT. Run All and download `CFP-FP-QA01-results.zip`.

The 55 visually flagged hashes and all 97 unresolved hashes are embedded from the AI-assisted review. No further manual review, R3 archive or original CFP dataset is required for this audit. No input files are changed. No data or results are uploaded.

## What we measure
1. Decode the exact original bytes with Pillow and OpenCV, checking agreement before any review resizing/re-encoding.
2. Save **lossless PNGs** of all 55 flagged crops and contact sheets. These establish whether the lack of detail exists in the source bin rather than only the review JPEGs.
3. Report image variation, edge strength and intensity range for **all** unique crops. These are descriptive texture measures, not a validated face detector or a rule for deleting images.
4. Recompute raw cosine pair scores from the frozen embeddings. Learn each FAR=0.001 threshold on nine original folds and apply it to the held-out fold.
5. At those **same frozen full-data thresholds**, compare all pairs, pairs touching the 55 flags, their complement, and corresponding groups for all 97 unresolved crops. TAR is the fraction of genuine pairs accepted; FAR is the fraction of impostor pairs accepted.

This is **post-hoc sensitivity analysis**: examining how an existing result depends on flagged inputs. Subset results have different denominators and selection bias. They are not replacement benchmark scores. We do not retrain thresholds on selected subsets, alter original E01–E04 results, infer person labels, or certify E05 identity mapping.

The reproduced E01 low-FAR counts are compared with the recorded 3070 genuine accepts and 5 impostor accepts. Any discrepancy is explicitly reported; it is not silently treated as a reproduced baseline.

In [ ]:
from pathlib import Path
import io, re, json, hashlib, pickle, zipfile, shutil, platform
from collections import defaultdict
import numpy as np
import pandas as pd
from PIL import Image
from IPython.display import display, FileLink

INPUT_ROOT = Path('/kaggle/input')
WORK_ROOT = Path('/kaggle/working') if Path('/kaggle').exists() else Path.cwd()
# Set explicit paths if more than one matching input exists. ZIP paths are accepted.
CACHE_INPUT = None
REFERENCE_INPUT = None   # directory or ZIP of person-labelled CFP images
BIN_INPUT = None         # optional exact cfp_fp.bin
REFERENCE_PROVENANCE = 'Kaggle chinafax/cfpw-dataset; source identity folders and FP protocol' # e.g. provider URL, release name and how obtained
# Standard CFP layout; change only if the source documents a different layout.
IDENTITY_PATTERN = r'(?:^|/)(?P<identity>\d{3})/(?P<pose>frontal|profile)/[^/]+$'
from datetime import datetime, timezone
OUT = WORK_ROOT / ('CFP-FP-QA01-' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%f'))
OUT.mkdir(parents=True, exist_ok=True)
# Remove stale success outputs from an earlier run, never user inputs.
for name in ['verified_mapping.csv', 'identity_folds.csv']:
    (OUT / name).unlink(missing_ok=True)
issues = []
input_records = []

def sha(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1024*1024), b''): h.update(block)
    return h.hexdigest()

def record(path):
    input_records.append({'path': str(path), 'sha256': sha(path), 'bytes': path.stat().st_size})

def unpack(path):
    path = Path(path)
    if path.is_dir(): return path
    if path.suffix.lower() != '.zip': raise ValueError('Expected directory or ZIP: '+str(path))
    record(path)
    dest = WORK_ROOT / 'mapping_inputs' / sha(path)[:16]
    dest.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(path) as z:
        if sum(i.file_size for i in z.infolist()) > 20*1024**3:
            raise ValueError('Archive exceeds 20 GiB extraction limit.')
        for item in z.infolist():
            target = (dest / item.filename).resolve()
            if not target.is_relative_to(dest.resolve()) or ((item.external_attr >> 16) & 0o170000) == 0o120000:
                raise ValueError('Unsafe ZIP entry: '+item.filename)
        z.extractall(dest)
    return dest

def pixel_hash(data):
    with Image.open(io.BytesIO(data)) as im:
        im = im.convert('RGB')
        return hashlib.sha256(str(im.size).encode()+b'RGB'+im.tobytes()).hexdigest()

print('Python', platform.python_version(), '| NumPy', np.__version__, '| pandas', pd.__version__)

DOWNLOAD_BASELINE_BIN = True
EXPECTED_BIN_SHA256 = '76306c783c2ef59c8569ebdcdd2f529f450bcc3fad57c94a5fc2b91df0f10370'
BASELINE_BIN_URL = ('https://huggingface.co/datasets/Icar/val_sets/resolve/'
                    '1f7800d9d5ba295554573a4e292e870b64a49b74/cfp_fp.bin')
# Downloads are checksum-checked before deserialization.
print('Results directory:', OUT)


In [ ]:
REVIEWER = 'ChatGPT — AI-assisted photographic-correspondence review (not human review)'
REVIEW_R3_HASH = '77ffc74c57965ab3b645164f5a8121d647ca3f9cde8ae6eb2bda6cb417547874'
FLAGGED_55 = set(['06a1a8975bc91357e40b160ef3b0b161ffddd3c8ce49e1efb670f2626fcd3830', '088378179dc606887e43e62d6a2c491cc6204d7adb91aa154e4ae50eff3d1132', '091a7e39a537c07ab84d8397fbb70b1788a86e411d8f606881b1c1f1d220f012', '097d80a8e8a4aaab8b20198de1e6e7bc9a17162152ed85693727d056bb4e6212', '11950361f998f439778b5c164a31621a88e4e3f960b13e62d24734c3b1926bdb', '145d5d632c3356ae9332c0748d62f955e6e26a5a2320c977fa3aec560851c457', '153bdb2fa5593c636cc825b5535cd4671204d38835222b892436ae4fe8b9dcb2', '233121d2eaf5ddfb517ce287f9668da599c0a71c68627051d1075825eef25605', '240f274b6f072bd94c027d87c1bc86311f44e4bc29a809ff49373ee11ffe3cbd', '293b488631eea7c99d79bac57895fad7edb500ba3178fe2575640f9a4ae6329b', '2a2babb4547ed2b1dc75e320c3be1d5ec858c2d54bec4d92bf3d37f7703d26cb', '2d67f1e3361b45e27b511d5686099fd3c4e5d8c7f7a7469eebe82fb7784cd55e', '3304352e3f5b585b9ee86eec15374320aee437f6799b19d5fd6746cba3a89b8c', '36846cac6e0a3a976077ea69121158ac747884654d2bc700d4ebfd8378f4a7ad', '3c8c73618eaad7c0764639a445198004d98125688b8aa65989d7447ff323d1ca', '3f6639a2ce0d4d1734b2d6e5fb71730c562347d7c5abed3a3189e0ac7b8f5e07', '3fd1f11a216607fc0d939d0d0008e293685129bb037e8956a762b2b53b400442', '44d0ecd649392f859e890a82477df9d213730bb737ed8c38e42b491312233ee4', '48bb48448a6861c7ed524a5d5337bf471401d59ff057351fe1a9f69c52bd54c3', '49776b29798f977d4b1194dbd8123bf385810e61db196bedb6bae4f3c29d9213', '4c050737f2bb3b373d252ca3a733e6df907b1fecdf59605121b22090a46a5bc0', '4eab88d72a59f57d71d5123ef131fa57ca1cd7f3b018186206531fc3def9ee34', '52446883f31bab095f9506ece6a471d55496f3cff65ac6b4d5baf26adc27f0fd', '52f0534e06613c92557febc35d091ca4f4691c5d85402222b2cfdd6b61445fdc', '52f3980143b8b457c9e2c072d977275baae9d083c0e8b43fd395529fb0bbe8f9', '583d849f2f5b473f0bcda9d20011f9ed6abf678bc9dafdba6a5af32cb21e9153', '5d8e651319f5b7bf807fa26faf5349120a0825cb32c872242c2639cc94940ee0', '663011b53e127ca8c5d4d63d34682c58bd5eb967a9cfba1472487e4c1161c142', '689904733e1e9f62abb856a8575b0b2c5639643a402fb6e01d33f0e84a67aeb4', '7b2d49aff7e02ef0f9f2d292066cff3fffb75fffc203ed787d5342e4d7918eb6', '817b3da2f0b0b1d9e9c81204f3a78db7996fc410cc1ecbf974e78c5f9537bea0', '886fae09aa3bfe969def87e49429f6c657a15d64659f9ca9692391afbd724eae', '9b0e325e4868f032a78ce035c4601c0de177f8022975fef30d6ad3bea22989fb', '9eb65e5e4148cf7e7ed6ffa3dc8f7b3a20b352380aed37dac287e5011d07a5c7', 'ac9e5daee27ec145312912e441222c53196f9b58cbe829ff8e153c38f906df46', 'ad9516a5a5541264784667877f58ad073c013998a73ea7d29d91c7777a28c444', 'ae8609c1ea080362846324be947a2c6260868fd122ba96664d6f37188c53b3d9', 'b45762044379456c39ea3e7a5c1f90425bb4a670dc2634b8f056d7a926465dc8', 'b5bf5c64950ce00696e4a6587b54994a89c6d1e3e3cedcc2c36f0e79736a779c', 'b683d9cb1bbcd2f4cc247871b02c409df58ad05e3426c9713d2586e30397c09e', 'b6c0e25f105569bc176476ea5253e65f33c7291f61dab8553a12601b9fda8911', 'b79434ec563362480cbe54a24034bf56a8e2d26c07ce51564535e696889a1a8a', 'b7c7830905769481bb72f6bec13d75c63bb346e1ad9ae5c61b9a13a6d18e0d36', 'b9e409e65aa795a1231e2f01f6500c04b8737f92ba934fb319101ec5f63e27a6', 'bbeb0009d425e48dd02239fc02f2abd8cbbc7dd744f044d4fb98263b95499d82', 'bda921110517bbb565f39eaaf55d37abfe7e8c1b450bf608c6d2d2edb4aa25d7', 'bfd18bbb8fd8f633583d42a7805392604f0e60c01fac7626563494f99699dec8', 'c35fd76a8b1486245f073ad7d2f90ffef2647a02ef6bc174a0be1f4b59774b8f', 'c8435d4c6822262bef3714aae35b5ad6866f33a990235e431685106692459083', 'cb019bdc4a0042bfad017ffd0c270af058e141baaa1587dbfe97c5fdea008080', 'cef704d7fdc61107ff721a4445b47969a3a193fbe5c4e2b31c8db7b941835b78', 'dc2b6d1e7546361c350e0157548c2723071df0d8399f99948726deeca87b9d7e', 'e33734d295e0a00defc2523d5ac65439431af5d62da54a182488ea3ca354373c', 'e56c276d73269741344750bbb00b3cd10d3eaea56c61781ae7af9e8038c136ff', 'e82c1726eec4dd4ec5547a49061e870213358d10a548b3f0b98a34faa2ac5c32'])
UNRESOLVED_97 = set(['016243a1cd09dc33561c7b9ed3516e42937be3e73d05948e66278ab187bf46b0', '06a1a8975bc91357e40b160ef3b0b161ffddd3c8ce49e1efb670f2626fcd3830', '088378179dc606887e43e62d6a2c491cc6204d7adb91aa154e4ae50eff3d1132', '091a7e39a537c07ab84d8397fbb70b1788a86e411d8f606881b1c1f1d220f012', '097d80a8e8a4aaab8b20198de1e6e7bc9a17162152ed85693727d056bb4e6212', '0fa729b65b2af378da201bf9f7e6690cacc04b7eae32fdd3a6865e9936cc0b6c', '11950361f998f439778b5c164a31621a88e4e3f960b13e62d24734c3b1926bdb', '145d5d632c3356ae9332c0748d62f955e6e26a5a2320c977fa3aec560851c457', '153bdb2fa5593c636cc825b5535cd4671204d38835222b892436ae4fe8b9dcb2', '1cbae7a945676e0443e6a85541f0463fbe9f02f8c122572d25e425a1324d0508', '233121d2eaf5ddfb517ce287f9668da599c0a71c68627051d1075825eef25605', '240f274b6f072bd94c027d87c1bc86311f44e4bc29a809ff49373ee11ffe3cbd', '25396a99c68fa2184c7d88dd46e46f3fd68b9c0684f0f257d2373211304686d7', '293b488631eea7c99d79bac57895fad7edb500ba3178fe2575640f9a4ae6329b', '2a2babb4547ed2b1dc75e320c3be1d5ec858c2d54bec4d92bf3d37f7703d26cb', '2aa57c6f1978488c419a4f476e9aa58c2537491daa0d458b75b66c0e57b6e656', '2d67f1e3361b45e27b511d5686099fd3c4e5d8c7f7a7469eebe82fb7784cd55e', '3304352e3f5b585b9ee86eec15374320aee437f6799b19d5fd6746cba3a89b8c', '36846cac6e0a3a976077ea69121158ac747884654d2bc700d4ebfd8378f4a7ad', '3acedbbaf9a8b7f4a66e22f2ff0564939a3398955507863038c3caaa7370799a', '3b62982fb8f7239d17c93f089878452592bdaadbde64ea7e3eb3f790812a09ef', '3c8c73618eaad7c0764639a445198004d98125688b8aa65989d7447ff323d1ca', '3f6639a2ce0d4d1734b2d6e5fb71730c562347d7c5abed3a3189e0ac7b8f5e07', '3fd1f11a216607fc0d939d0d0008e293685129bb037e8956a762b2b53b400442', '3fdf3b49e4cdc18e4ce60b816b64c9545e19a722f8485f932af6c90a5cc3d3c2', '416dff60c12b34024aeb05f5a9dc2a0662feabf528dceadd40900778e0446c0c', '43ffb272ba3d57debc553b88b90e0b7bd1a62487b7370ce56061059da78cf4cb', '44d0ecd649392f859e890a82477df9d213730bb737ed8c38e42b491312233ee4', '48bb48448a6861c7ed524a5d5337bf471401d59ff057351fe1a9f69c52bd54c3', '49776b29798f977d4b1194dbd8123bf385810e61db196bedb6bae4f3c29d9213', '4c050737f2bb3b373d252ca3a733e6df907b1fecdf59605121b22090a46a5bc0', '4cf0031778d1f3df2e75aa7cc467e9115066ef3c19c4388a6e7ca7b1000fc857', '4eab88d72a59f57d71d5123ef131fa57ca1cd7f3b018186206531fc3def9ee34', '4fa889e8f0f5d489613ec6757ed5c15977516668bd9b0d61f4c1c9240310350d', '52446883f31bab095f9506ece6a471d55496f3cff65ac6b4d5baf26adc27f0fd', '52f0534e06613c92557febc35d091ca4f4691c5d85402222b2cfdd6b61445fdc', '52f3980143b8b457c9e2c072d977275baae9d083c0e8b43fd395529fb0bbe8f9', '583d849f2f5b473f0bcda9d20011f9ed6abf678bc9dafdba6a5af32cb21e9153', '5d8e651319f5b7bf807fa26faf5349120a0825cb32c872242c2639cc94940ee0', '5ebbc6c74b58361640c1e7ce559ac9761fe06b8e55b351ac996db5d84eba7dde', '620d4c3ede316363ea68ddb572cf9e000fea2f85e56b29265f892d656c921d82', '64967929647aa209e47256e34c90c7bddf8b668d903ad0b3d6b38067e4d960e3', '663011b53e127ca8c5d4d63d34682c58bd5eb967a9cfba1472487e4c1161c142', '689904733e1e9f62abb856a8575b0b2c5639643a402fb6e01d33f0e84a67aeb4', '78132385da7029ac7dbe9a2082e3580cb221cba60e1c4b9cbe36b6f7818295a5', '793985a597a6edeade38049fe6363838f7b83503a9392f99c033186ad99e27f6', '794657f72b1c6454c485c5dc19b558813cea4e7584969f69131c5ea9efcc206f', '7b2d49aff7e02ef0f9f2d292066cff3fffb75fffc203ed787d5342e4d7918eb6', '817b3da2f0b0b1d9e9c81204f3a78db7996fc410cc1ecbf974e78c5f9537bea0', '886fae09aa3bfe969def87e49429f6c657a15d64659f9ca9692391afbd724eae', '89ff853e13ae7e72b31b2a974c2aa1ec5762b34a601c562586ed721c947a29be', '8ec9620a04139908d12948a28dc1d50ed252fe9e1b95a17b31d6592f63bd32e3', '9002fc00b13e15dcd5f154b123e689c1ed8fcf25b557d472076db3716f32fe7f', '94abee20fbc6daf3390ebcf951f72e5aa07f0169b0cd2cddb36b3fb2db4a0491', '9b0e325e4868f032a78ce035c4601c0de177f8022975fef30d6ad3bea22989fb', '9dbc283770ac8d29b3ba3f1f572ff7175f1e2486255ae22ac7550d42eb31205d', '9eb65e5e4148cf7e7ed6ffa3dc8f7b3a20b352380aed37dac287e5011d07a5c7', 'a0dba587accca2a7ce2d715241c4ef556638342b438877b545954b8cc5cf99df', 'a48417d447fb5de35e03d75bb92202ba3b8eb5057248434193673d038172c64d', 'a5b0c8dd67fcdd15d499b29935323198ae7790b940c6c4e2761cc18dd20e8bf5', 'a5f345430c989e40361cf384419992c085f035bd7b6bfd76390547572fde9aaf', 'a603840195e669aab391568dc3d3bd7324670e4d1b99fef5f23c53ef65b51f1c', 'a8807c118ddc846304908bebe43fbb3baa10bd3a7de369c05fc6daa904a63ce7', 'ac9e5daee27ec145312912e441222c53196f9b58cbe829ff8e153c38f906df46', 'ad9516a5a5541264784667877f58ad073c013998a73ea7d29d91c7777a28c444', 'ae8609c1ea080362846324be947a2c6260868fd122ba96664d6f37188c53b3d9', 'b45762044379456c39ea3e7a5c1f90425bb4a670dc2634b8f056d7a926465dc8', 'b5bf5c64950ce00696e4a6587b54994a89c6d1e3e3cedcc2c36f0e79736a779c', 'b683d9cb1bbcd2f4cc247871b02c409df58ad05e3426c9713d2586e30397c09e', 'b6c0e25f105569bc176476ea5253e65f33c7291f61dab8553a12601b9fda8911', 'b79434ec563362480cbe54a24034bf56a8e2d26c07ce51564535e696889a1a8a', 'b7c7830905769481bb72f6bec13d75c63bb346e1ad9ae5c61b9a13a6d18e0d36', 'b9683b14c2d91bc2a867cf485715b92004037b1cc70854f80d0824c56e7a7791', 'b9e409e65aa795a1231e2f01f6500c04b8737f92ba934fb319101ec5f63e27a6', 'bb6a9d979425f4f2ad9a20aa76f68a0c8e9b27612e9501958ebd9687b9bb7b28', 'bbeb0009d425e48dd02239fc02f2abd8cbbc7dd744f044d4fb98263b95499d82', 'bda921110517bbb565f39eaaf55d37abfe7e8c1b450bf608c6d2d2edb4aa25d7', 'bfd18bbb8fd8f633583d42a7805392604f0e60c01fac7626563494f99699dec8', 'c35fd76a8b1486245f073ad7d2f90ffef2647a02ef6bc174a0be1f4b59774b8f', 'c8435d4c6822262bef3714aae35b5ad6866f33a990235e431685106692459083', 'cb019bdc4a0042bfad017ffd0c270af058e141baaa1587dbfe97c5fdea008080', 'cef704d7fdc61107ff721a4445b47969a3a193fbe5c4e2b31c8db7b941835b78', 'd02b14347ca9f20f186187cbf4f555787efc8794df6afe3e3e80f66d79ce5e9d', 'dc2b6d1e7546361c350e0157548c2723071df0d8399f99948726deeca87b9d7e', 'e0899925465e9ccc8968ec3f46cb5fc7cc5a731064b53e1a48d434728479e099', 'e2d48aa59cecc0398573d33c63bd9b905c48104958b4113a79865487891b02ce', 'e33734d295e0a00defc2523d5ac65439431af5d62da54a182488ea3ca354373c', 'e4740eb8873a9bbd9ab144f2e49999c63e3fbe5c9686fe4f550baa5c5854d2cf', 'e4d0259b2c297d5e44ac730014b61d81e1f3b99fdcefdc7d8fe8b99b3117c4d5', 'e56c276d73269741344750bbb00b3cd10d3eaea56c61781ae7af9e8038c136ff', 'e82c1726eec4dd4ec5547a49061e870213358d10a548b3f0b98a34faa2ac5c32', 'eba9c347936d72a0dc0996b23f45a948f79d6c0d47db028abf2a84b7c821ce5d', 'ef60fba7fd056bfb6a7b29fbb50886b9613d287d753508082e35a22e69610172', 'f64630e9c14d84ebc111957911d10ab44bb24df4d213d83d7ff6823526bd9469', 'f7182c86ac44f33fb38b65cf0183d5c90d33f6d53ad4798f2bc54e3de04dbc3c', 'f82f271d020530481be9d660ab75fbb2d15c463cffee86f68d296e756c970285', 'f9b1c0804460f59382d09a8062d7cb7cb43694791ce9d019b68b927342bb593c'])
assert len(FLAGGED_55)==55 and len(UNRESOLVED_97)==97


## 1. Verify the frozen cache and restore the exact original bin

In [ ]:
if CACHE_INPUT is None:
    direct = sorted(p for p in INPUT_ROOT.rglob('aligned_pair_side_metadata.csv') if (p.parent/'aligned_pair_side_embeddings.npy').is_file()) if INPUT_ROOT.exists() else []
    archives = []
    if not direct and INPUT_ROOT.exists():
        for p in INPUT_ROOT.rglob('*.zip'):
            try:
                with zipfile.ZipFile(p) as z:
                    if {'aligned_pair_side_metadata.csv','aligned_pair_side_embeddings.npy'}.issubset({Path(x).name for x in z.namelist()}): archives.append(p)
            except zipfile.BadZipFile: pass
    choices = [p.parent for p in direct] or archives
    if len(choices) != 1:
        raise RuntimeError(f'Set CACHE_INPUT to the one R4 cache directory/ZIP. Candidates: {choices}')
    CACHE_INPUT = choices[0]
root = unpack(CACHE_INPUT)
paths = list(root.rglob('aligned_pair_side_metadata.csv'))
if len(paths) != 1: raise RuntimeError('Cache must contain exactly one pair-side metadata file.')
meta_path = paths[0]
meta = pd.read_csv(meta_path)
emb_path = meta_path.parent / 'aligned_pair_side_embeddings.npy'
record(meta_path); record(emb_path)
emb = np.load(emb_path, mmap_mode='r', allow_pickle=False)
required = {'bin_index','pair_id','side','fold','label','encoded_sha256','status'}
assert required.issubset(meta.columns), f'Missing columns: {required-set(meta.columns)}'
assert len(meta) == 14000 and emb.shape == (14000,512), 'Wrong R4 cache dimensions'
assert np.array_equal(meta.bin_index.to_numpy(), np.arange(14000)), 'Cache rows are not in bin order'
assert meta.status.eq('valid').all() and np.isfinite(emb).all(), 'Invalid embeddings'
assert np.allclose(np.linalg.norm(emb,axis=1),1,atol=1e-4), 'Non-normalized embeddings'
assert np.array_equal(meta.pair_id.to_numpy(),np.arange(14000)//2+1)
assert meta.side.tolist() == ['a','b']*7000
assert set(meta.label.unique()) == {0,1}
assert np.array_equal(meta.label.to_numpy()[::2],meta.label.to_numpy()[1::2])
assert meta.encoded_sha256.astype(str).str.fullmatch(r'[0-9a-f]{64}').all()
print(f'Cache valid: {len(meta):,} occurrences; {meta.encoded_sha256.nunique():,} unique encoded images.')


In [ ]:
pixel_by_encoded = {}
crop_bytes = {}
bin_verified = False
try:
    if BIN_INPUT is None:
        found = sorted(INPUT_ROOT.rglob('cfp_fp.bin'))
        if len(found) > 1: raise ValueError('Multiple bins attached; set BIN_INPUT explicitly.')
        if found: BIN_INPUT = found[0]
    if BIN_INPUT is None and DOWNLOAD_BASELINE_BIN:
        from urllib.request import urlopen
        destination = WORK_ROOT / 'cfp_fp_verified_source' / 'cfp_fp.bin'
        destination.parent.mkdir(parents=True, exist_ok=True)
        if not destination.exists() or sha(destination) != EXPECTED_BIN_SHA256:
            temporary = destination.with_suffix('.download')
            with urlopen(BASELINE_BIN_URL, timeout=60) as response, open(temporary,'wb') as f:
                shutil.copyfileobj(response, f)
            if sha(temporary) != EXPECTED_BIN_SHA256:
                temporary.unlink(missing_ok=True)
                raise ValueError('Downloaded bin checksum mismatch; file rejected.')
            temporary.replace(destination)
        BIN_INPUT = destination
    if BIN_INPUT is None: raise FileNotFoundError('No baseline bin selected; crop inspection unavailable.')
    path = Path(BIN_INPUT)
    if sha(path) != EXPECTED_BIN_SHA256: raise ValueError('Bin is not the exact R4 baseline file.')
    record(path)
    # Loading is permitted only after the exact baseline fingerprint passes.
    with open(path,'rb') as f: bins, bin_labels = pickle.load(f, encoding='bytes')
    if len(bins)!=14000 or len(bin_labels)!=7000: raise ValueError('Unexpected bin dimensions')
    if not np.array_equal(np.asarray(bin_labels,dtype=int),meta.label.to_numpy()[::2]):
        raise ValueError('Bin/cache labels differ')
    pending_pixels, pending_crops = {}, {}
    for idx, data in enumerate(bins):
        h = hashlib.sha256(data).hexdigest()
        if h != meta.encoded_sha256.iloc[idx]: raise ValueError(f'Bin/cache mismatch at occurrence {idx}')
        if h not in pending_pixels:
            pending_pixels[h] = pixel_hash(data)
            pending_crops[h] = data
    pixel_by_encoded, crop_bytes = pending_pixels, pending_crops
    bin_verified = True
    del bins, pending_pixels, pending_crops
    print('Exact R4 bin verified against all 14,000 cache rows.')
except Exception as exc:
    pixel_by_encoded, crop_bytes = {}, {}
    print('Bin unavailable or rejected:', type(exc).__name__, str(exc))
    issues.append('Baseline crop restoration: '+str(exc))

if not bin_verified: raise RuntimeError('Exact baseline bin is required for QA01. Enable Internet or attach the pinned file; see preceding message.')
assert FLAGGED_55.issubset(crop_bytes) and UNRESOLVED_97.issubset(crop_bytes)


## 2. Measure original pixels and export all 55 source crops

In [ ]:
import cv2
from PIL import ImageDraw,ImageOps

def texture_metrics(rgb):
    gray=cv2.cvtColor(rgb,cv2.COLOR_RGB2GRAY).astype(np.float64)
    bins=np.bincount(gray.astype(np.uint8).ravel(),minlength=256).astype(float)
    prob=bins[bins>0]/gray.size
    return dict(gray_std=float(gray.std()),range_p95_p05=float(np.percentile(gray,95)-np.percentile(gray,5)),
                entropy_bits=float(-(prob*np.log2(prob)).sum()),
                mean_gradient=float((np.abs(np.diff(gray,axis=0)).mean()+np.abs(np.diff(gray,axis=1)).mean())/2),
                black_fraction=float((gray<=3).mean()))

pixel_rows=[];direct=OUT/'original_flagged_png';direct.mkdir()
for h,data in crop_bytes.items():
    with Image.open(io.BytesIO(data)) as im:rgb=np.asarray(im.convert('RGB'))
    bgr=cv2.imdecode(np.frombuffer(data,dtype=np.uint8),cv2.IMREAD_COLOR)
    if bgr is None:raise ValueError('OpenCV decode failure: '+h)
    rgb_cv=cv2.cvtColor(bgr,cv2.COLOR_BGR2RGB)
    if rgb.shape!=rgb_cv.shape:raise ValueError('Decoder shape disagreement: '+h)
    difference=np.abs(rgb.astype(int)-rgb_cv.astype(int))
    pixel_rows.append(dict(encoded_sha256=h,review_flag_55=h in FLAGGED_55,
        unresolved_97=h in UNRESOLVED_97,width=rgb.shape[1],height=rgb.shape[0],
        decoder_max_abs_diff=int(difference.max()),decoder_mean_abs_diff=float(difference.mean()),
        **texture_metrics(rgb)))
    if h in FLAGGED_55:Image.fromarray(rgb).save(direct/(h+'.png'))
pixels=pd.DataFrame(pixel_rows)
pixels.to_csv(OUT/'all_crop_pixel_audit.csv',index=False)
pixels[pixels.review_flag_55].to_csv(OUT/'flagged_55_pixel_audit.csv',index=False)
pixels.groupby('review_flag_55')[['gray_std','range_p95_p05','entropy_bits','mean_gradient']].agg(['min','median','max']).to_csv(OUT/'texture_group_summary.csv')
for start in range(0,55,20):
    hashes=sorted(FLAGGED_55)[start:start+20]
    sheet=Image.new('RGB',(800,200*((len(hashes)+3)//4)),'white');draw=ImageDraw.Draw(sheet)
    for j,h in enumerate(hashes):
        x=(j%4)*200;y=(j//4)*200
        with Image.open(direct/(h+'.png')) as im:sheet.paste(im.resize((168,168),Image.Resampling.NEAREST),(x,y+25))
        draw.text((x+2,y+3),f'{start+j+1}: {h[:14]}',fill='black')
    sheet.save(OUT/f'original_flagged_sheet_{start//20+1}.png')
print('All 55 flagged crops saved directly from original bytes, without JPEG re-encoding.')
display(pixels.groupby('review_flag_55')[['gray_std','mean_gradient','entropy_bits']].median())


## 3. Reproduce full-data thresholds and retain every held-out decision

In [ ]:
def threshold_at_far(scores,labels,target=0.001):
    imp=np.sort(np.asarray(scores)[np.asarray(labels)==0])[::-1]
    allowed=int(np.floor(target*len(imp)))
    if not len(imp) or allowed>=len(imp):raise ValueError('Invalid threshold calibration inputs')
    return float(np.nextafter(np.float64(imp[allowed]),np.inf))

a=meta.iloc[::2].reset_index(drop=True);b=meta.iloc[1::2].reset_index(drop=True)
assert np.array_equal(a.label,b.label) and np.array_equal(a.fold,b.fold)
# Compute in float32, as in the cached baseline pipeline.
scores=np.sum(np.asarray(emb[::2],dtype=np.float32)*np.asarray(emb[1::2],dtype=np.float32),axis=1).astype(np.float64)
pairs=pd.DataFrame(dict(pair_id=a.pair_id,fold=a.fold,label=a.label,score=scores,
    touches_55=a.encoded_sha256.isin(FLAGGED_55)|b.encoded_sha256.isin(FLAGGED_55),
    touches_97=a.encoded_sha256.isin(UNRESOLVED_97)|b.encoded_sha256.isin(UNRESOLVED_97)))
pairs['threshold']=np.nan
calibration=[]
for fold in range(1,11):
    train=pairs.fold.ne(fold);test=~train
    assert int((test & pairs.label.eq(1)).sum())==350
    assert int((test & pairs.label.eq(0)).sum())==350
    t=threshold_at_far(pairs.loc[train,'score'].to_numpy(),pairs.loc[train,'label'].to_numpy())
    pairs.loc[test,'threshold']=t
    training_far=float((pairs.loc[train & pairs.label.eq(0),'score']>=t).mean())
    assert training_far<=0.001
    calibration.append(dict(fold=fold,threshold=t,training_far=training_far))
pairs['accepted']=pairs.score>=pairs.threshold
pairs.to_csv(OUT/'all_pair_decisions.csv',index=False)
pd.DataFrame(calibration).to_csv(OUT/'full_data_thresholds.csv',index=False)


## 4. Quantify contributions and sensitivity at unchanged thresholds
Every subset is descriptive. The complements are explicitly labelled, not presented as a repaired CFP benchmark. No quality exclusions are applied to original results.

In [ ]:
def count_metrics(frame):
    y=frame.label.to_numpy();p=frame.accepted.to_numpy()
    ng=int((y==1).sum());ni=int((y==0).sum())
    tp=int((p & (y==1)).sum());fp=int((p & (y==0)).sum())
    return dict(pairs=len(frame),genuine=ng,impostor=ni,tp=tp,fn=ng-tp,fp=fp,tn=ni-fp,
                tar=tp/ng if ng else None,far=fp/ni if ni else None)
cohorts={'all_original_pairs':np.ones(len(pairs),dtype=bool),
         'touches_review_flagged_55':pairs.touches_55,
         'complement_of_flagged_55':~pairs.touches_55,
         'touches_unresolved_97':pairs.touches_97,
         'complement_of_unresolved_97':~pairs.touches_97}
summary=pd.DataFrame([dict(cohort=name,**count_metrics(pairs[mask])) for name,mask in cohorts.items()])
summary.to_csv(OUT/'baseline_sensitivity.csv',index=False)
fold_rows=[dict(cohort=name,fold=f,**count_metrics(pairs[mask & pairs.fold.eq(f)]))
           for name,mask in cohorts.items() for f in range(1,11)]
pd.DataFrame(fold_rows).to_csv(OUT/'baseline_sensitivity_by_fold.csv',index=False)
full=count_metrics(pairs)
reproduced=full['tp']==3070 and full['fp']==5
print('Recorded E01 low-FAR counts reproduced:',reproduced)
display(summary)
report=dict(experiment='CFP-FP-QA01',run_completed=True,baseline_bin_verified=True,
    expected_bin_sha256=EXPECTED_BIN_SHA256,unique_crops=len(pixels),flagged_crops_exported=55,
    review_source='AI-assisted review of R4 rendered images',reviewer=REVIEWER,review_r3_report_sha256=REVIEW_R3_HASH,
    decoder_max_difference=int(pixels.decoder_max_abs_diff.max()),
    e01_counts_reproduced=reproduced,recorded_e01=dict(tp=3070,fp=5),observed_full_data=full,
    inputs=input_records,attacker_executed=False,original_results_modified=False,
    interpretation='Post-hoc quality audit and descriptive fixed-threshold sensitivity; no validated quality cutoff or definitive identity labels.',
    next_step='Inspect direct-bin flagged PNGs and quantify pair impact. Specify a documented quality/provenance policy before any E05 subset experiment; keep original benchmark results unchanged.')
(OUT/'quality_report.json').write_text(json.dumps(report,indent=2,allow_nan=False))
tracker=['CFP-FP-QA01','Check original-bin crop quality and baseline impact','Decoder agreement; affected pairs; fixed-threshold TAR/FAR',
         f"55 original crops exported; affected pairs={int(pairs.touches_55.sum())}; E01 counts reproduced={reproduced}",
         'Quality/sensitivity audit; no E05 attacker or automatic exclusions']
(OUT/'tracker_row.tsv').write_text('\t'.join(tracker)+'\n')
manifest=[dict(path=p.relative_to(OUT).as_posix(),sha256=sha(p)) for p in sorted(OUT.rglob('*')) if p.is_file() and p.name!='output_manifest.json']
(OUT/'output_manifest.json').write_text(json.dumps(manifest,indent=2))
archive=Path(shutil.make_archive(str(WORK_ROOT/'CFP-FP-QA01-results'),'zip',OUT))
print('Download:',archive)
import os
display(FileLink(os.path.relpath(archive,Path.cwd())))
